In [1]:
DATASET = "autompg"
SPLIT = 1
SEED = 0

LENGTHSCALE_INIT = 1.0
LENGTHSCALE_MIN = 1.0e-3
LENGTHSCALE_MAX = 1.0e3

NUM_PARTICLES = 32
ALPHA = 1.0
STEP_SIZE = 1.0e-4

SIGMA_INIT = 0.2
SIGMA_MIN = 0.1
SIGMA_MAX = 1.0
SIGMA_LR = 0.1
KERNEL_LR = 0.01

VAL_FRACTION = 0.25
ADAPT_TARGET = None
NUM_ADAPT_STEPS = 20000
WARMUP_STEPS = 1000
SIGMA_ADAPT_STEPS = 600
KERNEL_ADAPT_STEPS = 300

NUM_SAMPLE_STEPS = 3000
BURN_FRACTION = 0.9
THIN = 50

In [2]:
import jax
import jax.numpy as jnp
import jax.random as jr

import gpjax as gpx
import optax as ox
import paramax as px

from sklearn.preprocessing import StandardScaler

from non_parametric_pro import ula
from non_parametric_pro.ula import parametric_ula

from non_parametric_pro.density import ProParameters, pro_logdensity_fn
from non_parametric_pro.util import (
    cholesky_basis,
    nlpd_gp,
    nlpd_pro,
    prediction_basis,
    run_inference_algorithm_with_burn_in,
    train_val_split,
)
from non_parametric_pro.inducing import PointInducingBasis
from non_parametric_pro.parameter_adaptation import parameter_adaptation
from non_parametric_pro.data.uci.uci import load_uci_regression_dataset

jax.config.update("jax_enable_x64", True)

In [3]:
key = jr.PRNGKey(SEED)

example = load_uci_regression_dataset(DATASET, split=SPLIT)

scaler_x = StandardScaler()
scaler_y = StandardScaler()
x_train = scaler_x.fit_transform(example.x_train)
y_train = scaler_y.fit_transform(example.y_train)
x_test = scaler_x.transform(example.x_test)
y_test = scaler_y.transform(example.y_test)

print("N_train:", x_train.shape[0], " N_test:", x_test.shape[0], " D:", x_train.shape[1])

N_train: 313  N_test: 79  D: 7


In [4]:
D = x_train.shape[1]

data = gpx.Dataset(X=x_train, y=y_train)
exact_gp_kernel = gpx.kernels.RBF(lengthscale=LENGTHSCALE_INIT * jnp.sqrt(D) * jnp.ones((D,)))

prior = gpx.gps.Prior(mean_function=gpx.mean_functions.Zero(), kernel=exact_gp_kernel)
likelihood = gpx.likelihoods.Gaussian(num_datapoints=data.n)
posterior = prior * likelihood

gp_posterior, _ = gpx.fit_scipy(
    model=posterior,
    objective=lambda p, d: -gpx.objectives.conjugate_mll(p, d),
    train_data=data,
    verbose=False,
)

gp_latent_train_dist = gp_posterior.predict(x_train, train_data=data)
gp_predictive_train_dist = gp_posterior.likelihood(gp_latent_train_dist)

gp_predictive_train_mean = gp_predictive_train_dist.mean
gp_predictive_train_std = jnp.sqrt(gp_predictive_train_dist.variance)

gp_sigma = gp_posterior.likelihood.obs_stddev
gp_kernel = gp_posterior.prior.kernel

gp_latent_dist = gp_posterior.predict(x_test, train_data=data)
gp_predictive_dist = gp_posterior.likelihood(gp_latent_dist)

gp_predictive_mean = gp_predictive_dist.mean
gp_predictive_std = jnp.sqrt(gp_predictive_dist.variance)

In [5]:
kernel = gp_kernel
basis_full = cholesky_basis(kernel, x_train)
basis_dim = basis_full.shape[1]
row_selectable_basis = PointInducingBasis(z=x_train)

key, split_key, pos_key, adapt_key = jr.split(key, 4)
split = train_val_split(split_key, x_train, y_train, val_fraction=VAL_FRACTION)

fold_params = ProParameters(
    y=split.y_train,
    step_size=STEP_SIZE,
    sigma=gpx.parameters.SigmoidBounded(SIGMA_INIT, low=SIGMA_MIN, high=SIGMA_MAX),
    alpha=ALPHA,
)
initial_position = jr.normal(pos_key, (basis_dim, NUM_PARTICLES))

adaptation = parameter_adaptation(
    ula,
    pro_logdensity_fn,
    fold_params,
    x_train=split.x_train,
    initial_kernel=kernel,
    warmup_steps=WARMUP_STEPS,
    sigma_adapt_steps=SIGMA_ADAPT_STEPS,
    kernel_adapt_steps=KERNEL_ADAPT_STEPS,
    objective_fn=pro_logdensity_fn,
    inducing_basis=row_selectable_basis,
    x_val=split.x_val,
    y_val=split.y_val,
    adapt_target=ADAPT_TARGET,
    sigma_optimizer=ox.adam(SIGMA_LR),
    kernel_optimizer=ox.adam(KERNEL_LR),
    progress_bar=True,
)
adaptation_results, adaptation_info = adaptation.run(adapt_key, initial_position, num_steps=NUM_ADAPT_STEPS)
adapted_sigma = px.unwrap(adaptation_results.parameters.sigma)
print("Adapted sigma:", adapted_sigma)

Running parameter adaptation


<div><progress max="20000" value="20000"></progress> 100.00% [20000/20000 00:00&lt;?]</div>

Adapted sigma: 0.23174725885564396


In [6]:
if KERNEL_ADAPT_STEPS > 0:
    adapted_kernel = px.unwrap(jax.tree.map(lambda x: x[-1], adaptation_info.kernel))
    basis_full = cholesky_basis(adapted_kernel, x_train)
else:
    adapted_kernel = kernel

print("lengthscale:", px.unwrap(adapted_kernel.lengthscale))
print("variance:", px.unwrap(adapted_kernel.variance))

pro_params = ProParameters(
    y=y_train,
    basis=basis_full,
    step_size=STEP_SIZE,
    sigma=adaptation_results.parameters.sigma,
    alpha=ALPHA,
    residual_std=None,
)
algorithm = parametric_ula(pro_logdensity_fn, pro_params)
key, sample_key = jr.split(key)
_, (states, _) = run_inference_algorithm_with_burn_in(
    rng_key=sample_key,
    inference_algorithm=algorithm,
    num_steps=NUM_SAMPLE_STEPS,
    burn_ratio=BURN_FRACTION,
    initial_position=adaptation_results.state.position,
    progress_bar=True,
)
particles = states.position[::THIN]

lengthscale: [6.85237368e+04 3.18436065e+00 2.42547062e+00 3.45033938e+00
 7.91244361e+00 1.97821230e+00 2.28327396e+00]
variance: 1.5390220267769275


<div><progress max="2700" value="2700"></progress> 100.00% [2700/2700 00:00&lt;?]</div>

<div><progress max="300" value="300"></progress> 100.00% [300/300 00:00&lt;?]</div>

In [7]:
test_basis, test_cov = prediction_basis(
    adapted_kernel, x_train, x_train, pro_params
)
print("PRO Train NLPD:", nlpd_pro(y_train, test_basis, test_cov, particles, parameters=pro_params))

test_basis, test_cov = prediction_basis(
    adapted_kernel, x_train, x_test, pro_params
)

print("PRO Test NLPD:", nlpd_pro(y_test, test_basis, test_cov, particles, parameters=pro_params))
print("GP Train NLPD:", nlpd_gp(y_train, gp_predictive_train_mean, gp_predictive_train_std))
print("GP Test NLPD:", nlpd_gp(y_test, gp_predictive_mean, gp_predictive_std))

PRO Train NLPD: 0.008883532655479021
PRO Test NLPD: 0.2170922019745609
GP Train NLPD: 0.20753481265525023
GP Test NLPD: 0.29998152795232985
